# Clio-Infra at historical borders — what to download

Every Clio-Infra indicator is deposited on DataverseNL as its own dataset. This
lists all 86, with the page to open in a browser, grouped by how much the
historical borders can add, and ticks off the ones already filed.

**Where the list comes from.** DataverseNL cannot be browsed by script — it
refuses automated clients. It does not need to be. Each main-site workbook cites
its indicator by Handle (`10622/…`), and the Handle resolver redirects that to
the DataverseNL DOI; `utils.datasets.build()` reads only the redirect, never
DataverseNL itself. Input: `notebooks/clio/data/inventory.json`. Output:
`data/datasets.json` (gitignored). Re-running it skips DOIs already resolved.

**Filing a download.** In a browser, open the dataset page and use *Access Dataset →
Download ZIP* — the zip carries the working paper (`.docx`), which a single-file
download does not, and the paper is where the metadata is. Unzip it under
`data/raw/`, or file it with:

```python
accept.accept("~/Downloads/<folder>", version="1.1")
```

The DOI is found from the workbook's own title row; pass `doi=` only if it says
it cannot match. `version` is the one on the dataset page. The working paper's
own "Version" section is the authors' revision of the dataset, not this. The download
folder is copied, never moved.

In [1]:
import sys
sys.path.append("../")          # notebooks/clio_historical/, for utils

import json
import pandas
import yaml
from IPython.display import Markdown, display

from utils import accept, datasets, documentation, workbook

catalog = datasets.build()                      # cached after the first run
inventory = json.loads((datasets.CLIO_DATA / "inventory.json").read_text())

# What is filed is whatever sits under data/raw/, however it got there -- copied
# by hand or through accept(). A workbook's title row names its indicator (a
# paper's title where the workbook has none); a paper means the zip was used.
by_title = {" ".join(e["name"].split()).casefold(): slug for slug, e in catalog.items()}
by_title.update(documentation.WORKBOOK_ALIASES)
have_book = set()
for book in (accept.DATA_DIR / accept.RAW).glob("*/*.xlsx"):
    slug = by_title.get(" ".join(workbook.read(book)["title"].split()).casefold())
    if slug: have_book.add(slug)
papers_path = accept.DATA_DIR / documentation.DOCUMENTATION
papers = yaml.safe_load(papers_path.read_text()) if papers_path.exists() else {}
have_paper = {e["clio_slug"] for e in papers.values()}
have_book |= have_paper
print(f"{len(catalog)} datasets: {len(have_paper)} with a working paper, "
      f"{len(have_book - have_paper)} workbook only, {len(catalog) - len(have_book)} not filed")


86 datasets: 52 with a working paper, 21 workbook only, 13 not filed


## Three groups

Historical borders only change data **before 1946** — after that the deposit's
CShapes rows reproduce the main site. And the main-site survey found two kinds
of indicator where borders matter little: tables filled with zeros for every
country (conflicts, mining), and model reconstructions at benchmark years
(Agriculture, Biodiversity).

1. **First** — measured, with data before 1946. Where the borders change the story.
2. **Later** — the filled tables and reconstructions. Worth it for the unit and
   the working paper; the borders add little.
3. **Last** — indicators that start after 1946. Unit and working paper only.

In [2]:
def group(slug):
    ind, entry = inventory[slug], catalog[slug]
    if entry["decades_before_1946"] == 0:
        return "3 last"
    if (ind["zero_share"] > 0.5 or ind["distinct_values"] <= 25
            or ind["category"] == "Agriculture" or slug == "Biodiversitynaturalness"):
        return "2 later"
    return "1 first"

rows = [{"group": group(slug), "category": e["category"], "indicator": e["name"],
         "span": f"{e['start']}–{e['end']}", "doi": e["doi"], "page": e["page"],
         "filed": "✓ paper" if slug in have_paper else "workbook" if slug in have_book else ""}
        for slug, e in catalog.items()]
table = pandas.DataFrame(rows).sort_values(["group", "category", "indicator"])
table.groupby("group").size().rename("datasets").to_frame()

,datasets
group,
1 first,44
2 later,38
3 last,4


In [3]:
def checklist(frame):
    lines = ["| | category | indicator | span | dataset |", "| --- | --- | --- | --- | --- |"]
    for r in frame.itertuples():
        lines.append(f"| {r.filed} | {r.category} | {r.indicator} | {r.span} | [{r.doi}]({r.page}) |")
    return "\n".join(lines)

for name, frame in table.groupby("group"):
    display(Markdown(f"### {name[2:].capitalize()} — {len(frame)}\n\n" + checklist(frame)))

### First — 44

| | category | indicator | span | dataset |
| --- | --- | --- | --- | --- |
| ✓ paper | Demography | Composite Measure  of Wellbeing | 1820–2000 | [10.34894/NH52ZW](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/NH52ZW) |
| ✓ paper | Demography | Female life Expectancy at Birth | 1750–2000 | [10.34894/AFNO6I](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/AFNO6I) |
|  | Demography | Global Extreme Poverty (CBN) | 1820–2018 | [10.34894/GH1KUH](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/GH1KUH) |
|  | Demography | Global Extreme Poverty (DAD) | 1820–2018 | [10.34894/LVAKCN](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/LVAKCN) |
| ✓ paper | Demography | Global Hunger | 1820–2018 | [10.34894/KNTMVP](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/KNTMVP) |
| workbook | Demography | Height | 1550–2000 | [10.34894/CGMOIO](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/CGMOIO) |
| workbook | Demography | Height Gini | 1810–1980 | [10.34894/DGKIQ8](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/DGKIQ8) |
| workbook | Demography | Infant Mortality | 1810–2000 | [10.34894/JITFDV](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/JITFDV) |
| ✓ paper | Demography | Life Expectancy at Birth (Total) | 1543–2012 | [10.34894/RFRR6D](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/RFRR6D) |
| ✓ paper | Demography | Male life Expectancy at Birth | 1750–2000 | [10.34894/QJRBX9](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/QJRBX9) |
| ✓ paper | Demography | Total Population | 1500–2000 | [10.34894/HW27BU](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/HW27BU) |
| ✓ paper | Demography | Total Urban Population | 1500–2000 | [10.34894/VFBJND](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/VFBJND) |
| ✓ paper | Demography | Urbanization Ratio | 1500–2000 | [10.34894/YESEB2](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/YESEB2) |
| workbook | Environment | CO2 Emissions per Capita | 1750–2010 | [10.34894/9WCWSP](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/9WCWSP) |
| workbook | Environment | SO2 Emissions per Capita | 1850–2000 | [10.34894/XRYW0G](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/XRYW0G) |
|  | Environment | Total CO2 Emissions | 1750–2008 | [10.34894/EPURQH](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/EPURQH) |
|  | Environment | Total SO2 Emissions | 1850–2000 | [10.34894/PSB2G6](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/PSB2G6) |
| ✓ paper | Finance | Exchange Rates to UK Pound | 1500–2013 | [10.34894/5NLBWR](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/5NLBWR) |
| ✓ paper | Finance | Exchange Rates to US Dollar | 1500–2013 | [10.34894/NBIFGP](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/NBIFGP) |
| ✓ paper | Finance | Long-Term Government  Bond Yield | 1727–2011 | [10.34894/139GLM](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/139GLM) |
| ✓ paper | Finance | Total Gross Central Government  Debt as a Percentage of GDP | 1692–2010 | [10.34894/1VZHBY](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/1VZHBY) |
| workbook | Gender Equality | Gender Equality of Numeracy | 1810–1960 | [10.34894/TVL1RB](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/TVL1RB) |
| ✓ paper | Human Capital | Average Years of Education | 1820–2010 | [10.34894/KV6ASD](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/KV6ASD) |
| ✓ paper | Human Capital | Book Titles per Capita | 1500–2009 | [10.34894/H3ZKE8](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/H3ZKE8) |
| ✓ paper | Human Capital | Educational Inequality Gini  Coefficient | 1850–2010 | [10.34894/YKCEZS](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/YKCEZS) |
| workbook | Human Capital | Numeracy (Total) | 1500–1970 | [10.34894/I0L2TH](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/I0L2TH) |
| ✓ paper | Institutions | Homicide Rates | 1800–2010 | [10.34894/F2XDR7](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/F2XDR7) |
| ✓ paper | Institutions | Latent Democracy Variable | 1850–2000 | [10.34894/DBYA1J](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/DBYA1J) |
| ✓ paper | Institutions | Political Competition | 1810–2011 | [10.34894/RAGSNW](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/RAGSNW) |
| ✓ paper | Institutions | Political Participation | 1810–2000 | [10.34894/MHMFGA](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/MHMFGA) |
| ✓ paper | Institutions | Polyarchy | 1810–2010 | [10.34894/TBQIMW](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/TBQIMW) |
| ✓ paper | Labour Relations | Number of Days Lost in  Labour Disputes | 1927–2013 | [10.34894/PNGMA0](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/PNGMA0) |
|  | Labour Relations | Number of Labour Disputes | 1927–2013 | [10.34894/YTIILL](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/YTIILL) |
| ✓ paper | Labour Relations | Number of Workers Involved  in Labour Disputes | 1927–2013 | [10.34894/MQ6VBA](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/MQ6VBA) |
| ✓ paper | Labour Relations | Working week  in manufacturing | 1800–2008 | [10.34894/Z2TEHC](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/Z2TEHC) |
| ✓ paper | National Accounts | GDP per Capita | 1500–2016 | [10.34894/I7YZIV](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/I7YZIV) |
| ✓ paper | National Accounts | Social Spending | 1820–2016 | [10.34894/WEUEXA](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/WEUEXA) |
| ✓ paper | Prices and Wages | Income Inequality | 1820–2000 | [10.34894/H4HNSX](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/H4HNSX) |
| ✓ paper | Prices and Wages | Inflation | 1500–2010 | [10.34894/MMUGQL](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/MMUGQL) |
| ✓ paper | Prices and Wages | Labourers Real Wage | 1820–2008 | [10.34894/UFVNXT](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/UFVNXT) |
| workbook | Prices and Wages | Wealth Decadal Ginis | 1820–2010 | [10.34894/LALT8Y](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/LALT8Y) |
|  | Prices and Wages | Wealth Top10 percent share | 1820–2010 | [10.34894/AYMP5W](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/AYMP5W) |
| workbook | Prices and Wages | Wealth Total | 1820–2010 | [10.34894/K48NNT](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/K48NNT) |
| workbook | Prices and Wages | Wealth Yearly Ginis | 1820–2015 | [10.34894/STBWLJ](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/STBWLJ) |

### Later — 38

| | category | indicator | span | dataset |
| --- | --- | --- | --- | --- |
|  | Agriculture | Cattle per Capita | 1500–2010 | [10.34894/XPD1RI](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/XPD1RI) |
|  | Agriculture | Cropland per Capita | 1500–2010 | [10.34894/IM0MQN](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/IM0MQN) |
|  | Agriculture | Goats per Capita | 1500–2010 | [10.34894/YQRHTR](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/YQRHTR) |
|  | Agriculture | Pasture per Capita | 1500–2010 | [10.34894/NEP7BX](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/NEP7BX) |
|  | Agriculture | Pigs per Capita | 1500–2010 | [10.34894/BP3TTE](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/BP3TTE) |
|  | Agriculture | Sheep per Capita | 1500–2010 | [10.34894/PRA4VH](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/PRA4VH) |
| workbook | Agriculture | Total Cattle | 1500–2010 | [10.34894/WOK2TI](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/WOK2TI) |
| workbook | Agriculture | Total Cropland | 1500–2010 | [10.34894/GK0VJA](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/GK0VJA) |
| workbook | Agriculture | Total Number of Goats | 1500–2010 | [10.34894/RVLYZ1](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/RVLYZ1) |
| workbook | Agriculture | Total Number of Pigs | 1500–2010 | [10.34894/3SS1IG](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/3SS1IG) |
| workbook | Agriculture | Total Number of Sheep | 1500–2010 | [10.34894/AHHX4A](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/AHHX4A) |
| workbook | Agriculture | Total Pasture | 1500–2010 | [10.34894/KZFK0B](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/KZFK0B) |
| workbook | Environment | Biodiversity - naturalness | 1500–2010 | [10.34894/2NEIXG](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/2NEIXG) |
| ✓ paper | Finance | Gold Standard | 1800–2010 | [10.34894/0PLMSY](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/0PLMSY) |
| workbook | Gender Equality | Gender-equal Inheritance | 1920–2000 | [10.34894/0ZG6Q3](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/0ZG6Q3) |
| workbook | Gender Equality | Sex Ratio | 1900–2000 | [10.34894/90EPKO](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/90EPKO) |
| ✓ paper | Human Capital | Universities Founded | 1502–2013 | [10.34894/OB6RSY](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/OB6RSY) |
| ✓ paper | Institutions | Armed Conflicts (Internal) | 1500–2000 | [10.34894/ZWKAH6](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/ZWKAH6) |
| ✓ paper | Institutions | Armed Conflicts (International) | 1500–2000 | [10.34894/KEBYOQ](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/KEBYOQ) |
| ✓ paper | Institutions | Competitiveness of Executive  Recruitment (XRCOMP) | 1800–2012 | [10.34894/PE0ZPK](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/PE0ZPK) |
| ✓ paper | Institutions | Competitiveness of Participations  (PARCOMP) | 1800–2012 | [10.34894/5Q1MFZ](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/5Q1MFZ) |
| ✓ paper | Institutions | Executive Constraints  (XCONST) | 1800–2012 | [10.34894/4M3SAP](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/4M3SAP) |
| ✓ paper | Institutions | Openness of Executive  Recruitment (XROPEN) | 1800–2012 | [10.34894/ORSVIN](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/ORSVIN) |
| ✓ paper | Institutions | Polity2 Index | 1800–2012 | [10.34894/NSARFE](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/NSARFE) |
| ✓ paper | Institutions | Regulation of Chief Executive  Recruitment (XRREG) | 1800–2012 | [10.34894/GLVTIU](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/GLVTIU) |
|  | Institutions | Regulation of Participation  (PARREG) | 1800–2012 | [10.34894/H3QTGJ](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/H3QTGJ) |
| ✓ paper | Production | Aluminium Production | 1850–2012 | [10.34894/E10G8W](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/E10G8W) |
| ✓ paper | Production | Bauxite Production | 1880–2012 | [10.34894/OL0R4D](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/OL0R4D) |
| ✓ paper | Production | Copper Production | 1700–2012 | [10.34894/RKJVZB](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/RKJVZB) |
| ✓ paper | Production | Gold Production | 1681–2012 | [10.34894/P79BO2](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/P79BO2) |
| ✓ paper | Production | Iron Ore Production | 1820–2012 | [10.34894/VPQO6C](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/VPQO6C) |
| ✓ paper | Production | Lead Production | 1705–2012 | [10.34894/SOU6DV](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/SOU6DV) |
| ✓ paper | Production | Manganese Production | 1835–2012 | [10.34894/BUWLSA](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/BUWLSA) |
| ✓ paper | Production | Nickel Production | 1850–2012 | [10.34894/FTYKZL](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/FTYKZL) |
| ✓ paper | Production | Silver Production | 1681–2012 | [10.34894/LZHYDP](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/LZHYDP) |
| ✓ paper | Production | Tin Production | 1700–2012 | [10.34894/QRXGSJ](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/QRXGSJ) |
| ✓ paper | Production | Tungsten Production | 1860–2012 | [10.34894/VI2SCV](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/VI2SCV) |
| ✓ paper | Production | Zinc Production | 1705–2012 | [10.34894/ZZSQAM](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/ZZSQAM) |

### Last — 4

| | category | indicator | span | dataset |
| --- | --- | --- | --- | --- |
| workbook | Gender Equality | Gender Equality Years  of Education | 1950–2000 | [10.34894/U7XDMU](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/U7XDMU) |
| workbook | Gender Equality | Historical Gender Equality Index | 1950–2003 | [10.34894/73GOFX](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/73GOFX) |
| ✓ paper | Gender Equality | Share of Women in Parliament | 1960–2010 | [10.34894/FLAPZY](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/FLAPZY) |
| ✓ paper | Institutions | Unified Democracy Scores  (UDS) | 1946–2008 | [10.34894/SGX07R](https://dataverse.nl/dataset.xhtml?persistentId=doi:10.34894/SGX07R) |